# Overall
- This notebook includes the feature importance analyses and selection by using the coefficients scores 
- coefficients scores calculates by using Lasso and RandomForestRegressor

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.base import BaseEstimator, TransformerMixin
import lightgbm as lgb
from sklearn.metrics import mean_squared_error

# 1. Veri Yükleme
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test_x.csv')

# Hedef değişken ve bağımsız değişkenlerin ayrılması
X = train_df.drop(columns=['career_success_score', 'student_id'])
y = train_df['career_success_score']
X_test = test_df.drop(columns=['student_id'], errors='ignore')

# 2. Özel Özellik Mühendisliği (Feature Engineering) Sınıfı
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # Mentor feedback metin uzunluğu
        X['feedback_length'] = X['mentor_feedback_text'].fillna("").apply(len)
        X['feedback_word_count'] = X['mentor_feedback_text'].fillna("").apply(lambda x: len(x.split()))
        
        # Akademik ve Teknik Skor Oranları / Kombinasyonları
        X['technical_score_avg'] = X[['coding_score', 'problem_solving_score', 'data_structures_score', 'sql_score', 'machine_learning_score']].mean(axis=1)
        X['project_impact'] = X['project_quality_score'] * (X['real_client_project_count'] + 1)
        X['internship_efficiency'] = X['internship_duration_months'] / (X['internship_count'] + 1)
        X['hackathon_success'] = X['hackathon_awards'] / (X['hackathon_count'] + 1)
        X['github_activity'] = X['github_repo_count'] * X['github_avg_stars']
        X['interview_success_rate'] = X['interviews_attended'] / (X['applications_sent'] + 1)
        X['soft_skills_avg'] = X[['communication_score', 'teamwork_score', 'leadership_score', 'presentation_score']].mean(axis=1)
        
        return X

# 3. Sütun Tiplerinin Belirlenmesi
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_features.remove('mentor_feedback_text') # Metin sütununu çıkarıyoruz, özel işleyeceğiz.
# Yeni eklenen mühendislik özelliklerini sayısal listeye ekleyelim
new_numeric_features = [
    'feedback_length', 'feedback_word_count', 'technical_score_avg', 
    'project_impact', 'internship_efficiency', 'hackathon_success', 
    'github_activity', 'interview_success_rate', 'soft_skills_avg'
]
numeric_features.extend(new_numeric_features)

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_features.remove('mentor_feedback_text')

# 4. Alt Pipeline'ların Oluşturulması
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

text_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ('tfidf', TfidfVectorizer(max_features=1000, stop_words=None)), # Türkçe stopwords eklenebilir
    ('svd', TruncatedSVD(n_components=10, random_state=42))
])

# 5. Ana Preprocessor (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('text', text_transformer, ['mentor_feedback_text'])
    ])

# 6. Tam Pipeline (Özellik Mühendisliği + Preprocessor + Model)
pipeline = Pipeline(steps=[
    ('feature_engineer', FeatureEngineer()),
    ('preprocessor', preprocessor),
    ('model', lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42))
])

# 7. Doğrulama ve Eğitim (Train-Test Split)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline.fit(X_train, y_train)

# Doğrulama Kümse Tahmini ve Metrik
y_pred = pipeline.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"Validation RMSE: {rmse:.4f}")

# 8. Test Verisi Üzerinden Tahmin Yapma
test_predictions = pipeline.predict(X_test)

# Sonuçları kaydetme
submission = pd.DataFrame({
    'student_id': test_df['student_id'],
    'career_success_score': test_predictions
})
submission.to_csv('submission.csv', index=False)
print("Tahminler 'submission.csv' dosyasına kaydedildi.")

ValueError: list.remove(x): x not in list

In [7]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import TruncatedSVD

# 1. Verileri Yükleme
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test_x.csv')

# Hedef ve Özelliklerin Ayrılması
X = train_df.drop('career_success_score', axis=1)
y = train_df['career_success_score']

# 2. Özellik Gruplarını Tanımlama
numeric_features = [col for col in X.columns if X[col].dtype in ['float64', 'int64']]
categorical_features = ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']
text_feature = 'mentor_feedback_text'

# 3. Transformer'lar
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Mentor metinleri için: TF-IDF + Boyut İndirgeme (SVD)
text_transformer = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(max_features=500, stop_words='english')),
    ('svd', TruncatedSVD(n_components=50)) 
])

# 4. Ana Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('text', text_transformer, text_feature)
    ])

# 5. Model Pipeline'ı
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Eğitim
pipeline.fit(X, y)

# Tahmin
predictions = pipeline.predict(test_df)

In [9]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import cross_val_score

# 1. Ham Verileri Yükleme
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test_x.csv')

# Metin verilerindeki eksiklikleri pipeline öncesi doldurma
train_df['mentor_feedback_text'] = train_df['mentor_feedback_text'].fillna('no_feedback')
test_df['mentor_feedback_text'] = test_df['mentor_feedback_text'].fillna('no_feedback')

# Bağımsız değişkenler (X) ve Hedef değişken (y)
X = train_df.drop(['career_success_score', 'student_id'], axis=1)
y = train_df['career_success_score']
X_test = test_df.drop('student_id', axis=1)

# 2. Değişken Gruplarını Belirleme
categorical_features = ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']
numeric_features = [col for col in X.columns if X[col].dtype in ['float64', 'int64'] and col not in categorical_features]
text_feature = 'mentor_feedback_text'

# 3. Pipeline Transformer'larını Hazırlama
# Sayısal veriler için: Medyan ile doldur ve Ölçeklendir
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Kategorik veriler için: En sık geçen ile doldur ve One-Hot Encode yap
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Metin verisi için: TF-IDF ile kelime vektörleri oluştur ve 20 boyuta indirge (SVD)
text_transformer = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(max_features=100, stop_words='english')),
    ('svd', TruncatedSVD(n_components=20))
])

# 4. Sütun Dönüştürücü (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('text', text_transformer, text_feature)
    ])

# 5. Ana Model Pipeline'ı
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1))
])

# 6. Çapraz Doğrulama (Cross Validation) Skoru
scores = cross_val_score(pipeline, X, y, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
print(f"Cross-Validation RMSE Skoru: {-scores.mean():.4f}")

# 7. Tüm veriyle eğitimi tamamlama ve Test setini tahmin etme
pipeline.fit(X, y)
predictions = pipeline.predict(X_test)

# 8. Tahminleri CSV olarak kaydetme
submission = pd.DataFrame({
    'student_id': test_df['student_id'],
    'career_success_score': predictions
})
submission.to_csv('predictions.csv', index=False)

Cross-Validation RMSE Skoru: 10.0336


# Mixed approach

In [11]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

# Veri yükleme (Dosya yolları düzeltildi)
train_df = pd.read_csv('data/train.csv')

# Pipeline öncesi metinleri temizleme
train_df['mentor_feedback_text'] = train_df['mentor_feedback_text'].fillna('no_feedback')

X_train = train_df.drop(['career_success_score', 'student_id'], axis=1)
y_train = train_df['career_success_score']

# Dönüştürücü sınıfı
class GroupImputer(BaseEstimator, TransformerMixin):
    def __init__(self, group_col, target_col):
        self.group_col = group_col
        self.target_col = target_col
        self.medians_ = {}
        self.global_median_ = 0

    def fit(self, X, y=None):
        self.medians_ = X.groupby(self.group_col)[self.target_col].median().to_dict()
        self.global_median_ = X[self.target_col].median()
        return self

    def transform(self, X):
        X_transformed = X.copy()
        mapped_medians = X_transformed[self.group_col].map(self.medians_).fillna(self.global_median_)
        X_transformed[self.target_col] = X_transformed[self.target_col].fillna(mapped_medians)
        return X_transformed

# Özellik grupları
mean_features = ['english_exam_score', 'linkedin_profile_score', 'hr_interview_score', 'portfolio_score']
median_features = ['github_avg_stars', 'open_source_contribution_count']
categorical_features = ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']
text_feature = 'mentor_feedback_text'

all_defined = set(mean_features + median_features + categorical_features + [text_feature, 'internship_duration_months', 'internship_count'])
other_numeric_features = [col for col in X_train.columns if col not in all_defined and X_train[col].dtype in ['float64', 'int64']]

# Transformerlar
mean_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

median_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

other_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

text_transformer = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(max_features=100, stop_words='english')),
    ('svd', TruncatedSVD(n_components=20))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('mean_num', mean_transformer, mean_features),
        ('median_num', median_transformer, median_features),
        ('other_num', other_numeric_transformer, other_numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('text', text_transformer, text_feature),
        ('pass', 'passthrough', ['internship_duration_months']) 
    ])

# Pipeline'lar
rf_pipeline = Pipeline(steps=[
    ('group_imputer', GroupImputer(group_col='internship_count', target_col='internship_duration_months')),
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))
])

lr_pipeline = Pipeline(steps=[
    ('group_imputer', GroupImputer(group_col='internship_count', target_col='internship_duration_months')),
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# CV Hesaplama
rf_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
lr_scores = cross_val_score(lr_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)

print(f"Random Forest CV RMSE: {-rf_scores.mean():.4f}")
print(f"Linear Regression CV RMSE: {-lr_scores.mean():.4f}")

Random Forest CV RMSE: 10.0685
Linear Regression CV RMSE: 9.4863


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor # XGBoost'u dahil ediyoruz

# --- 1. Veri Yükleme ve Ön Temizlik ---
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test_x.csv')

train_df['mentor_feedback_text'] = train_df['mentor_feedback_text'].fillna('no_feedback')
test_df['mentor_feedback_text'] = test_df['mentor_feedback_text'].fillna('no_feedback')

X_train = train_df.drop(['career_success_score', 'student_id'], axis=1)
y_train = train_df['career_success_score']
X_test = test_df.drop('student_id', axis=1)

# --- 2. Özel Dönüştürücü Sınıfı (Senin Domain Mantığın) ---
class GroupImputer(BaseEstimator, TransformerMixin):
    def __init__(self, group_col, target_col):
        self.group_col = group_col
        self.target_col = target_col
        self.medians_ = {}
        self.global_median_ = 0

    def fit(self, X, y=None):
        self.medians_ = X.groupby(self.group_col)[self.target_col].median().to_dict()
        self.global_median_ = X[self.target_col].median()
        return self

    def transform(self, X):
        X_transformed = X.copy()
        mapped_medians = X_transformed[self.group_col].map(self.medians_).fillna(self.global_median_)
        X_transformed[self.target_col] = X_transformed[self.target_col].fillna(mapped_medians)
        return X_transformed

# --- 3. Özellik Grupları ---
mean_features = ['english_exam_score', 'linkedin_profile_score', 'hr_interview_score', 'portfolio_score']
median_features = ['github_avg_stars', 'open_source_contribution_count']
categorical_features = ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']
text_feature = 'mentor_feedback_text'

# Diğer tüm sayısal değişkenleri otomatik bul
all_defined = set(mean_features + median_features + categorical_features + [text_feature, 'internship_duration_months', 'internship_count'])
other_numeric_features = [col for col in X_train.columns if col not in all_defined and X_train[col].dtype in ['float64', 'int64']]

# --- 4. Preprocessor (Veri Hazırlama Boru Hattı) ---
preprocessor = ColumnTransformer(
    transformers=[
        ('mean_num', Pipeline([('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())]), mean_features),
        ('median_num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), median_features),
        ('other_num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), other_numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
        ('text', Pipeline([('tfidf', TfidfVectorizer(max_features=100, stop_words='english')), ('svd', TruncatedSVD(n_components=20))]), text_feature),
        ('pass', 'passthrough', ['internship_duration_months']) # GroupImputer'dan gelen veri
    ])

# --- 5. Model Pipeline'larını Kurma ---

# Model A: Linear Regression (Önceki şampiyonumuz)
lr_pipeline = Pipeline(steps=[
    ('group_imputer', GroupImputer(group_col='internship_count', target_col='internship_duration_months')),
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Model B: XGBoost (Yeni güçlü adayımız)
xgb_pipeline = Pipeline(steps=[
    ('group_imputer', GroupImputer(group_col='internship_count', target_col='internship_duration_months')),
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, random_state=42, n_jobs=-1))
])

# --- 6. Modelleri Karşılaştırma (Cross-Validation) ---
print("Modeller değerlendiriliyor. Lütfen bekleyin...")

lr_scores = cross_val_score(lr_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
print(f"Linear Regression CV RMSE: {-lr_scores.mean():.4f}")

xgb_scores = cross_val_score(xgb_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
print(f"XGBoost CV RMSE:           {-xgb_scores.mean():.4f}")

# --- 7. Tüm Veri ile Eğitim ve Tahmin ---
print("\nTüm veriyle eğitim yapılıyor ve test seti tahmin ediliyor...")

# Linear Regression Tahminleri
lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)

# XGBoost Tahminleri
xgb_pipeline.fit(X_train, y_train)
xgb_preds = xgb_pipeline.predict(X_test)

# --- 8. Submission Dosyalarını Oluşturma ---
submission_lr = pd.DataFrame({'student_id': test_df['student_id'], 'career_success_score': lr_preds})
submission_xgb = pd.DataFrame({'student_id': test_df['student_id'], 'career_success_score': xgb_preds})

submission_lr.to_csv('submission_linear_regression.csv', index=False)
submission_xgb.to_csv('submission_xgboost.csv', index=False)

print("İşlem tamamlandı! 'submission_linear_regression.csv' ve 'submission_xgboost.csv' dosyaları hazır.")

Modeller değerlendiriliyor. Lütfen bekleyin...
Linear Regression CV RMSE: 9.4757
XGBoost CV RMSE:           9.1691

Tüm veriyle eğitim yapılıyor ve test seti tahmin ediliyor...
İşlem tamamlandı! 'submission_linear_regression.csv' ve 'submission_xgboost.csv' dosyaları hazır.


In [15]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

# 1. Veriyi okuma
train_df = pd.read_csv('data/train.csv') # Yol klasörüne göre 'train.csv' de olabilir
test_df = pd.read_csv('data/test_x.csv')

# Pipeline öncesi metinleri temizleme
train_df['mentor_feedback_text'] = train_df['mentor_feedback_text'].fillna('no_feedback')
test_df['mentor_feedback_text'] = test_df['mentor_feedback_text'].fillna('no_feedback')

X_train = train_df.drop(['career_success_score', 'student_id'], axis=1)
y_train = train_df['career_success_score']

# 2. Özel Dönüştürücü
class GroupImputer(BaseEstimator, TransformerMixin):
    def __init__(self, group_col, target_col):
        self.group_col = group_col
        self.target_col = target_col
        self.medians_ = {}
        self.global_median_ = 0

    def fit(self, X, y=None):
        self.medians_ = X.groupby(self.group_col)[self.target_col].median().to_dict()
        self.global_median_ = X[self.target_col].median()
        return self

    def transform(self, X):
        X_transformed = X.copy()
        mapped_medians = X_transformed[self.group_col].map(self.medians_).fillna(self.global_median_)
        X_transformed[self.target_col] = X_transformed[self.target_col].fillna(mapped_medians)
        return X_transformed

# 3. Özellik Grupları
mean_features = ['english_exam_score', 'linkedin_profile_score', 'hr_interview_score', 'portfolio_score']
median_features = ['github_avg_stars', 'open_source_contribution_count']
categorical_features = ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']
text_feature = 'mentor_feedback_text'

all_defined = set(mean_features + median_features + categorical_features + [text_feature, 'internship_duration_months', 'internship_count'])
other_numeric_features = [col for col in X_train.columns if col not in all_defined and X_train[col].dtype in ['float64', 'int64']]

# 4. Sütun Dönüştürücü
preprocessor = ColumnTransformer(
    transformers=[
        ('mean_num', Pipeline([('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())]), mean_features),
        ('median_num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), median_features),
        ('other_num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), other_numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
        ('text', Pipeline([('tfidf', TfidfVectorizer(max_features=100, stop_words='english')), ('svd', TruncatedSVD(n_components=20))]), text_feature),
        ('pass', 'passthrough', ['internship_duration_months'])
    ])

# 5. Yeni ve Güvenli XGBoost Pipeline
# Parametreler modelin ezber yapmasını (overfitting) önleyecek şekilde ayarlandı
xgb_pipeline_tuned = Pipeline(steps=[
    ('group_imputer', GroupImputer(group_col='internship_count', target_col='internship_duration_months')),
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=1.5,
        random_state=42,
        n_jobs=-1
    ))
])

# 6. CROSS VALIDATION (ÇAPRAZ DOĞRULAMA) SKORUNU HESAPLAMA
print("XGBoost Modeli 5 Katmanlı Çapraz Doğrulamadan geçiyor. Lütfen bekleyin...")
scores = cross_val_score(xgb_pipeline_tuned, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
print(f"\nOptimize Edilmiş XGBoost CV RMSE Skoru: {-scores.mean():.4f}")

# 7. Nihai Modeli Eğit ve Test Setini Tahmin Et
xgb_pipeline_tuned.fit(X_train, y_train)
X_test = test_df.drop('student_id', axis=1)
xgb_preds = xgb_pipeline_tuned.predict(X_test)

# --- 8. KRİTİK GÜVENLİK KONTROLLERİ ---
# Eğer model NaN (boş) tahmin ürettiyse onu train'in ortalamasıyla doldur:
xgb_preds = np.nan_to_num(xgb_preds, nan=y_train.mean())

# Eğer model 100'den büyük veya 0'dan küçük tahmin yaptıysa onları 0-100 sınırlarına çek:
xgb_preds = np.clip(xgb_preds, 0, 100)

# 9. Çıktı Alma (index=False ÇOK ÖNEMLİ)
sub = pd.DataFrame({
    'student_id': test_df['student_id'], 
    'career_success_score': xgb_preds
})

sub.to_csv('submission_xgboost_tuned_safe.csv', index=False)
print("\nGüvenli submission_xgboost_tuned_safe.csv dosyası başarıyla oluşturuldu!")

XGBoost Modeli 5 Katmanlı Çapraz Doğrulamadan geçiyor. Lütfen bekleyin...

Optimize Edilmiş XGBoost CV RMSE Skoru: 9.1691

Güvenli submission_xgboost_tuned_safe.csv dosyası başarıyla oluşturuldu!


In [16]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

# 1. VERİ YÜKLEME
# Burada _preprocessed dosyalarını kullanıyoruz çünkü sen orada Label Encoding yapmışsın anladığım kadarıyla
train_df = pd.read_csv('data/train.csv') 
test_df = pd.read_csv('data/test_x.csv')

# ÖNEMLİ: student_id'yi index yapalım ki satır kayması İMKANSIZ olsun
train_df.set_index('student_id', inplace=True)
test_df.set_index('student_id', inplace=True)

# Boş metinleri doldurma
train_df['mentor_feedback_text'] = train_df['mentor_feedback_text'].fillna('no_feedback')
test_df['mentor_feedback_text'] = test_df['mentor_feedback_text'].fillna('no_feedback')

X_train = train_df.drop('career_success_score', axis=1)
y_train = train_df['career_success_score']
X_test = test_df

# 2. SENİN FEATURE ENGINEERING ADIMIN (Custom Transformer Olarak)
class CustomFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        df = X.copy()
        
        # Senin yazdığın "add_features" ve "feature_engineering" mantıkları:
        df["years_since_graduation"] = df["application_year"] - df["graduation_year"]
        df["age_at_graduation"] = df["age"] - df["years_since_graduation"]
        df["is_recent_graduate"] = (df["years_since_graduation"] <= 1).astype(int)
        
        # Mülakat Başarı Oranı (0'a bölünme hatasını önlemek için +1)
        df["interview_conversion_rate"] = df["interviews_attended"] / (df["applications_sent"] + 1)
        
        # Hackathon Başarı Oranı
        df["hackathon_success_rate"] = df["hackathon_awards"] / (df["hackathon_count"] + 1)
        
        # Github Etki Skoru
        df["github_impact_score"] = (df["github_repo_count"] * 0.5 + 
                                     df["github_avg_stars"].fillna(0) * 0.2 + 
                                     df["open_source_contribution_count"].fillna(0) * 0.3)
        
        # Görünürlük (Visibility) İndeksi
        df["visibility_index"] = (df["linkedin_profile_score"].fillna(0) * 0.4 + 
                                  df["portfolio_score"].fillna(0) * 0.3 + 
                                  df["github_impact_score"] * 0.3)
        
        # Üniversite Tier (Label encoded olduğu için sayısala çevrildiyse veya kategorikse hata vermemesi için)
        # Eğer university_tier "Tier 1", "Tier 2" şeklindeyse rakamı çekelim, yoksa olduğu gibi alalım
        if df['university_tier'].dtype == 'O': # Object/String ise
             tier_num = df['university_tier'].str.extract(r'(\d+)').astype(float).fillna(3)
             df["academic_index"] = (df["cgpa"] * 0.5 + df["english_exam_score"].fillna(0) * 0.2 + 
                                     df["attendance_rate"] * 0.1 + (5 - tier_num[0]) * 0.1 - 
                                     df["failed_courses_count"] * 0.1)
        
        # Sonsuz (inf) değerleri temizle
        df = df.replace([np.inf, -np.inf], np.nan)
        return df

# 3. İŞLEM GRUPLARI VE SÜTUNLAR (Oluşturduğumuz yeni sütunları da ekliyoruz)
mean_features = ['english_exam_score', 'linkedin_profile_score', 'hr_interview_score', 'portfolio_score', 
                 'interview_conversion_rate', 'hackathon_success_rate', 'github_impact_score', 'visibility_index']
median_features = ['github_avg_stars', 'open_source_contribution_count']
categorical_features = ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']
text_feature = 'mentor_feedback_text'

# Yeni eklenenlerle beraber geriye kalan tüm sayısal özellikleri bulalım
temp_df = CustomFeatureEngineer().transform(X_train) # Gelecek sütunları görmek için dummy transform
all_defined = set(mean_features + median_features + categorical_features + [text_feature, 'internship_duration_months', 'internship_count'])
other_numeric_features = [col for col in temp_df.columns if col not in all_defined and temp_df[col].dtype in ['float64', 'int64']]

# 4. PREPROCESSOR
preprocessor = ColumnTransformer(
    transformers=[
        ('mean_num', Pipeline([('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())]), mean_features),
        ('median_num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), median_features),
        ('other_num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), other_numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
        ('text', Pipeline([('tfidf', TfidfVectorizer(max_features=100, stop_words='english')), ('svd', TruncatedSVD(n_components=20))]), text_feature),
        ('pass', 'passthrough', ['internship_duration_months']) 
    ])

# 5. NİHAİ PİPELİNE (Senin Feature Engineering -> Benim Preprocessor -> Model)
xgb_pipeline = Pipeline(steps=[
    ('feature_engineer', CustomFeatureEngineer()),
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=400, learning_rate=0.03, max_depth=5, 
        subsample=0.8, colsample_bytree=0.8, 
        reg_alpha=0.5, reg_lambda=1.5, random_state=42, n_jobs=-1
    ))
])

# 6. ÇAPRAZ DOĞRULAMA (MODEL PERFORMANS KONTROLÜ)
print("Senin Feature Engineering mantığınla XGBoost değerlendiriliyor...")
scores = cross_val_score(xgb_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
print(f"Yeni Özel CV RMSE Skoru: {-scores.mean():.4f}")

# 7. EĞİTİM VE TAHMİN
xgb_pipeline.fit(X_train, y_train)
xgb_preds = xgb_pipeline.predict(X_test)

# Güvenlik Kırpması (0-100 Sınırı)
xgb_preds = np.clip(xgb_preds, 0, 100)

# 8. GÜVENLİ SUBMISSION (SIRALAMA KAYMASI İMKANSIZ)
# Index'i student_id yaptığımız için doğrudan X_test'in index'ini alıyoruz.
sub = pd.DataFrame({
    'student_id': X_test.index, 
    'career_success_score': xgb_preds
})

sub.to_csv('submission_xgboost_FE_safe.csv', index=False)
print("\nGüvenli submission_xgboost_FE_safe.csv dosyası oluşturuldu! Artık sıralama hatası olmayacak.")

Senin Feature Engineering mantığınla XGBoost değerlendiriliyor...
Yeni Özel CV RMSE Skoru: 9.1140

Güvenli submission_xgboost_FE_safe.csv dosyası oluşturuldu! Artık sıralama hatası olmayacak.
